# Data Integrity

In [ ]:
# General TODOs:

# TODO: ...

# General NOTEs:

# NOTE: ...

# Setup

## Imports

In [ ]:
import hashlib
import warnings
from collections import Counter, defaultdict
from datetime import time
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import pandas as pd

from thesis_project import (
    collectors,
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "data-integrity"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
plt.style.use(config.MPL_CONFIG)

# Warning settings
warnings.simplefilter("always", UserWarning)

In [ ]:
# Sample date for quick analysis
year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"
fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

## Utilities

In [ ]:
def count_unique_lobs(root: Path) -> int:
    """
    Count the number of unique LOBs in a directory.
    """
    files = [p.name for p in root.rglob("*.parquet")]
    unique_files = set(files)
    return len(unique_files)


def dups_by_name(root: Path) -> List[str]:
    """
    Check for duplicate file names in a directory.
    """
    files = [p.name for p in root.rglob("*.parquet")]
    dups = [name for name, c in Counter(files).items() if c > 1]

    return dups


def dups_by_hash(root: Path) -> List[List[Path]]:
    """
    Check for duplicate files by grouping them by size and by hash.
    """

    def file_hash(path: Path, chunk_size: int = 1 << 20) -> str:
        h = hashlib.md5()
        with open(path, "rb") as f:
            while chunk := f.read(chunk_size):
                h.update(chunk)
        return h.hexdigest()

    size_map = defaultdict(list)
    for p in root.rglob("*.parquet"):
        size_map[p.stat().st_size].append(p)
    dups = []

    for paths in size_map.values():
        if len(paths) < 2:
            continue

        hash_map = defaultdict(list)
        for p in paths:
            hash_map[file_hash(p)].append(p)

        for group in hash_map.values():
            if len(group) > 1:
                dups.append(group)

    return dups


def time_to_seconds(t: time, cutoff: time = time(1, 0, 0)) -> int:
    s = t.hour * 3600 + t.minute * 60 + t.second
    cutoff_s = cutoff.hour * 3600 + cutoff.minute * 60 + cutoff.second
    if s < cutoff_s:
        s += 24 * 3600
    return s

# Main

In [ ]:
# Define collectors functions
run_collectors_single = collectors.run_collectors_single
run_collectors_double = collectors.run_collectors_double

# Define specific collectors
extractors = {
    "Empty LOBs": collectors.collect_empty_lobs,
    "NaNs": collectors.collect_nans,
    "Timestamps": collectors.collect_timestamps,
    "FUT specific": collectors.collect_fut_integrity,
    "CTD specific": collectors.collect_ctd_integrity,
}

## Duplicates

Quick check for duplicates in the datasets.

In [ ]:
# Count unique files
fut_unique = count_unique_lobs(LOB_FUT_DIR)
ctd_unique = count_unique_lobs(LOB_CTD_DIR)

# Check for duplicates by hash
fut_dups_name = dups_by_name(LOB_FUT_DIR)
ctd_dups_name = dups_by_name(LOB_CTD_DIR)

# Check for duplicates by hash
fut_dups_hash = dups_by_hash(LOB_FUT_DIR)
ctd_dups_hash = dups_by_hash(LOB_CTD_DIR)

fut_summary = pd.DataFrame(
    {
        "Unique LOBs": fut_unique,
        "Number of duplicates by name": len(fut_dups_name),
        "Number of duplicates by hash": len(fut_dups_hash),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "Unique LOBs": ctd_unique,
        "Number of duplicates by name": len(ctd_dups_name),
        "Number of duplicates by hash": len(ctd_dups_hash),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} duplicates summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
}
display(summary)
display(summary.attrs)

# filename = "duplicates_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## Futures specific issues

Analyze future LOBs after market closing 

In [ ]:
key = "FUT specific"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(00, 00, 00),
    end_time=time(23, 59, 59),
)

In [ ]:
fut_df = fut_results[key]
display(fut_df)

## CTD specific issues

Analyze CTD LOBs first row

## Empty LOBs
 Analyze empty LOBs in the dataset.

In [ ]:
key = "Empty LOBs"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.FUT_MRKT_OPEN,
    end_time=config.FUT_MRKT_CLOSE,
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
)

In [ ]:
fut_df = fut_results[key]
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures empty LOBs",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.FUT_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.FUT_MRKT_CLOSE.strftime("%H:%M"),
}

ctd_df = ctd_results[key]
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd empty LOBs",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.CTD_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.CTD_MRKT_CLOSE.strftime("%H:%M"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "empty_lobs.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
fut_summary = pd.DataFrame(
    {
        "File processed": len(fut_df),
        "Empty LOBs": fut_df["Empty"].sum(),
        "Empty LOBs (%)": fut_df["Empty"].mean() * 100,
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "File processed": len(ctd_df),
        "Empty LOBs": ctd_df["Empty"].sum(),
        "Empty LOBs (%)": ctd_df["Empty"].mean() * 100,
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} empty LOBs summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": config.FUT_MRKT_OPEN.strftime("%H:%M"),
    "Fut end time": config.FUT_MRKT_CLOSE.strftime("%H:%M"),
    "CTD start time": config.CTD_MRKT_OPEN.strftime("%H:%M"),
    "CTD end time": config.CTD_MRKT_CLOSE.strftime("%H:%M"),
}
display(summary)
display(summary.attrs)

# filename = "empty_lobs_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

## NaNs Analysis
Analyze NaNs in the dataset.

In [ ]:
key = "NaNs"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.FUT_MRKT_OPEN,
    end_time=config.FUT_MRKT_CLOSE,
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
)

In [ ]:
fut_df = fut_results[key]
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures NaNs analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.FUT_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.FUT_MRKT_CLOSE.strftime("%H:%M"),
}

ctd_df = ctd_results[key]
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd NaNs analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": config.CTD_MRKT_OPEN.strftime("%H:%M"),
    "End time": config.CTD_MRKT_CLOSE.strftime("%H:%M"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "nans.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
# Show top days with all NaNs on the rows
display(fut_df.sort_values(by="Rows all NaNs", ascending=False).head())
display(ctd_df.sort_values(by="Rows all NaNs", ascending=False).head())

In [ ]:
# Show top days with any NaNs on the rows
display(fut_df.sort_values(by="Rows any NaNs", ascending=False).head())
display(ctd_df.sort_values(by="Rows any NaNs", ascending=False).head())

In [ ]:
# Show top days with all NaNs on the cols
display(fut_df.sort_values(by="Cols all NaNs", ascending=False).head())
display(ctd_df.sort_values(by="Cols all NaNs", ascending=False).head())

## Timestamps Analysis

Analyze relevant timestastamps in the datasets.

In [ ]:
key = "Timestamps"
func = extractors[key]
extractor = {key: func}

fut_results = run_collectors_single(
    base_path=LOB_FUT_DIR,
    lob_type="futures",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(0, 0, 0),
    end_time=time(23, 59, 59),
)

ctd_results = run_collectors_single(
    base_path=LOB_CTD_DIR,
    lob_type="ctd",
    extractors=extractor,
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=time(0, 0, 0),
    end_time=time(23, 59, 59),
)

In [ ]:
fut_df = fut_results[key]
fut_df.attrs = {
    "Title": f"{TICKER.upper()} futures timestamps analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "FUT",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

ctd_df = ctd_results[key]
ctd_df.attrs = {
    "Title": f"{TICKER.upper()} ctd timestamps analysis",
    "Ticker": TICKER.upper(),
    "Lob type": "CTD",
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Start time": time(0, 0, 0).strftime("%H:%M:%S"),
    "End time": time(23, 59, 59).strftime("%H:%M:%S"),
}

display(fut_df)
display(fut_df.attrs)
display(ctd_df)
display(ctd_df.attrs)

# filename = "timestamps.parquet"
# fut_df.to_parquet(OUT_INT_DIR / f"fut_{filename}")
# ctd_df.to_parquet(OUT_INT_DIR / f"ctd_{filename}")

In [ ]:
fut_summary = pd.DataFrame(
    {
        "Min opening time": fut_df["First timestamp"].min(),
        "Min opening time (non-empty)": fut_df["First non-empty timestamp"].min(),
        "Max opening time": fut_df["First timestamp"].max(),
        "Max opening time (non-empty)": fut_df["First non-empty timestamp"].max(),
        "Min closing time": min(fut_df["Last timestamp"], key=time_to_seconds),
        "Min closing time (non-empty)": fut_df["Last non-empty timestamp"].min(),
        "Max closing time": fut_df["Last timestamp"].max(),
        "Max closing time (non-empty)": fut_df["Last non-empty timestamp"].max(),
    },
    index=["FUT"],
)

ctd_summary = pd.DataFrame(
    {
        "Min opening time": ctd_df["First timestamp"].min(),
        "Min opening time (non-empty)": ctd_df["First non-empty timestamp"].min(),
        "Max opening time": ctd_df["First timestamp"].max(),
        "Max opening time (non-empty)": ctd_df["First non-empty timestamp"].max(),
        "Min closing time": ctd_df["Last timestamp"].min(),
        "Min closing time (non-empty)": ctd_df["Last non-empty timestamp"].min(),
        "Max closing time": ctd_df["Last timestamp"].max(),
        "Max closing time (non-empty)": ctd_df["Last non-empty timestamp"].max(),
    },
    index=["CTD"],
)

summary = pd.concat([fut_summary, ctd_summary])
summary.index.name = "LOB type"
summary.attrs = {
    "Title": f"{TICKER.upper()} relevant timestamps summary",
    "Ticker": TICKER.upper(),
    "Start year": str(config.FBTP_START_YEAR),
    "End year": str(config.FBTP_END_YEAR),
    "Fut start time": "00:00:00",
    "Fut end time": "23:59:59",
    "CTD start time": "00:00:00",
    "CTD end time": "23:59:59",
}
display(summary)
display(summary.attrs)

# filename = "relevant_timestamps_summary.parquet"
# summary.to_parquet(OUT_PRO_DIR / filename)

### Plots

In [ ]:
import numpy as np
from matplotlib.ticker import FuncFormatter, MultipleLocator, PercentFormatter

lob_type = "fut"
filename = f"{lob_type}_timestamps.parquet"
df = pd.read_parquet(OUT_INT_DIR / f"{filename}")
timestamp_col = "Last timestamp"
figure_name = f"{lob_type}_hist_{timestamp_col.lower().replace(' ', '_')}"

bin_start_hh_mm = "14:00"
bin_width_seconds = 450
bin_width_title = f"{bin_width_seconds // 60:02d}:{bin_width_seconds % 60:02d}"
major_tick_minutes = 30
minor_tick_minutes = bin_width_seconds / 60
rotate_x_labels = 45
bar_alpha = 0.7
bar_edgecolor = "black"
minor_grid_alpha = 0.15
minor_grid_linestyle = ":"


ts_parsed = pd.to_datetime(
    df[timestamp_col].astype(str), format="%H:%M:%S", errors="coerce"
).dropna()
x = ts_parsed.dt.hour * 60 + ts_parsed.dt.minute + ts_parsed.dt.second / 60.0

start_h, start_m = map(int, bin_start_hh_mm.split(":"))
start_min = start_h * 60 + start_m
bin_width_min = bin_width_seconds / 60.0
end_min = np.ceil(x.max() / bin_width_min) * bin_width_min
bin_edges = np.arange(start_min, end_min + bin_width_min, bin_width_min)


def min_to_hh_mm(minutes, pos):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"


x_formatter = FuncFormatter(min_to_hh_mm)
y_formatter = PercentFormatter(1)

fig, ax = plt.subplots()
ax.hist(
    x,
    bins=bin_edges,  # type: ignore
    weights=np.full(len(x), 1 / len(x)),
    edgecolor=bar_edgecolor,
    alpha=bar_alpha,
)

ax.set_title(
    f"{lob_type.upper()} distribution of {timestamp_col.replace('_', ' ')}\n"
    f"{len(x)} samples, {x.nunique()} unique values\n"
    f"{bin_width_title} bins from {bin_start_hh_mm}:00"
)
ax.set_xlabel("Time (hh:mm)")
ax.set_ylabel("Frequency (%)")

ax.xaxis.set_major_formatter(x_formatter)
ax.xaxis.set_major_locator(MultipleLocator(major_tick_minutes))
ax.xaxis.set_minor_locator(MultipleLocator(minor_tick_minutes))
ax.yaxis.set_major_formatter(y_formatter)

ax.tick_params(axis="x", rotation=rotate_x_labels)
ax.grid(True, which="minor", alpha=minor_grid_alpha, linestyle=minor_grid_linestyle)

plt.show()

# save_figure(fig, figure_name)